# Statistical Methods in Imaging (SMI) Conference, 2026.
# Empowering Large Language Models with Statistics: A Practical Tutorial for Medical Imaging
**Ernest (Khashayar) Namdar, Dominik A. Deniffel, Pascal Tyrrell**

This tutorial focuses on classifying Acute Ischemic Stroke (AIS) from Radiology reports. We will use the `Label` and `Text` columns for a binary classification task.

Several similar pipelines were discussed in our publication:
```bibtex
@inproceedings{10.1117/12.3084682,
author = {Khashayar Namdar and Saeidehsadat Mirjalili and Lauren Erdman and Dominik A. Deniffel and Keith Brunt and Leo Anthony Celi},
title = {{Comparative evaluation of machine learning and large language model pipelines for identifying acute ischemic stroke in radiology reports}},
volume = {13926},
booktitle = {Medical Imaging 2026: Computer-Aided Diagnosis},
editor = {Axel Wism{\"u}ller and Thomas Martin Deserno},
organization = {International Society for Optics and Photonics},
publisher = {SPIE},
pages = {139261S},
keywords = {Stroke, NLP, Machine Learning, Large Language Models},
year = {2026},
doi = {10.1117/12.3084682},
URL = {https://doi.org/10.1117/12.3084682}
}
```

Also, the source for the dataset is:
```bibtex
@article{10.1371/journal.pone.0212778,
    doi = {10.1371/journal.pone.0212778},
    author = {Kim, Chulho AND Zhu, Vivienne AND Obeid, Jihad AND Lenert, Leslie},
    journal = {PLOS ONE},
    publisher = {Public Library of Science},
    title = {Natural language processing and machine learning algorithm to identify brain MRI reports with acute ischemic stroke},
    year = {2019},
    month = {02},
    volume = {14},
    url = {https://doi.org/10.1371/journal.pone.0212778},
    pages = {1-13},
    number = {2},
}
```


## Acute Ischemic Stroke Classification with Large Language Models
In this section, we will load our custom dataset `AIS.csv` and use the language model to extract whether the report indicates evidence of an acute ischemic stroke (yes/no).

In [ ]:
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from tqdm import tqdm
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

torch.random.manual_seed(0)

model_name = "microsoft/MediPhi"
# Initialize the Large Language Model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="cuda",          # Automatically loads the model weights onto the GPU for faster inference
    torch_dtype="auto",         # Uses the optimal precision (e.g., float16) to save GPU memory
    trust_remote_code=True,     # Required for some newer HuggingFace models with custom architectures
)
# The tokenizer converts raw text into numerical tokens the model can understand
tokenizer = AutoTokenizer.from_pretrained(model_name)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

generation_args = {
    "max_new_tokens": 10,
    "return_full_text": False,
    "do_sample": False,
}

def build_prompt(text):
    return f"""You are a medical AI assistant trained to classify radiology MRI reports.
Determine whether there is evidence of **acute ischemic stroke** in the following report.
Reply with a single word only: "yes" or "no".

Report:
---
{text}
---

Answer:"""


In [ ]:
import numpy as np
import gc

# Load Data
df = pd.read_csv('../data/AIS.csv')

# Chop into 10 subtasks
# Chop the dataset into smaller subtasks to prevent memory bloat over thousands of iterations
num_chunks = 10
df_chunks = np.array_split(df, num_chunks)

all_responses = []
all_ids = []

for i, chunk in enumerate(df_chunks):
    print(f"Processing subtask {i+1}/{num_chunks}...")
    chunk_responses = []
    chunk_ids = []
    
    for index, row in tqdm(chunk.iterrows(), total=len(chunk)):
        text = row['Text']
        prompt = build_prompt(text)
        messages = [
            {"role": "system", "content": "You are a radiologist."},
            {"role": "user", "content": prompt},
        ]
        
        output = pipe(messages, **generation_args)
        response = output[0]['generated_text'].strip().lower()
        
        # Save the whole response for the chunk
        chunk_responses.append(response)
        chunk_ids.append(row['ID'])
        
    all_responses.extend(chunk_responses)
    all_ids.extend(chunk_ids)
    
    # Aggregating intermediate results to disk
    temp_df = pd.DataFrame({'Id': chunk_ids, 'Response': chunk_responses})
    temp_df.to_csv(f'../results/LM_inference_chunk_{i+1}.csv', index=False)
    
    # Clear Python garbage and PyTorch GPU cache to sustain inference speed and prevent OOM errors
    gc.collect()
    torch.cuda.empty_cache()

# Aggregation and Parsing
all_inferences = []
for response in all_responses:
    if "yes" in response:
        inference = "yes"
    elif "no" in response:
        inference = "no"
    else:
        inference = "unknown"
    all_inferences.append(inference)

# Save inferences to CSV
results_df = pd.DataFrame({'Id': all_ids, 'Response': all_responses, 'Inference': all_inferences})
results_df.to_csv('../results/LM_inference.csv', index=False)
print("Aggregated inferences saved to ../results/LM_inference.csv")


In [5]:
# Evaluate Performance Metrics
# Map 'yes' to 1 and 'no' to 0 for evaluation
results_df['Pred'] = results_df['Inference'].map({'yes': 1, 'no': 0}).fillna(0)
y_true = df['Label']
y_pred = results_df['Pred']

acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

# Confusion matrix returns tn, fp, fn, tp
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

print(f"Accuracy:    {acc:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"F1 Score:    {f1:.4f}")


Accuracy:    0.9289
Sensitivity: 0.8542
Specificity: 0.9414
F1 Score:    0.7744
